# Đánh giá thực nghiệm Trợ lý RAG & Điểm danh Sinh trắc học (Bảng 3.14)

**Phạm vi kiểm chứng & Nguồn dữ liệu thực tế:**
1. **Phân hệ Trợ lý kỹ thuật RAG (`eam_app/algorithms/rag_chatbot/` + PostgreSQL `eam_db`):**
   - `LifeSafetyClassifier` (`classify_safety_context`): Kiểm chứng trên bộ 20 truy vấn kiểm thử có gán nhãn (10 tình huống nguy cấp thuộc 5 nhóm nguy cơ + 10 truy vấn phủ định an toàn / lý thuyết).
   - `Slang Query Expansion` (`expand_slang_query`): Kiểm chứng số mục từ trong `SLANG_TECHNICAL_DICTIONARY`, tỷ lệ kích hoạt và độ trễ mở rộng truy vấn. **Không công bố mức tăng Recall/độ bao phủ (như 34.2% trước đây)** vì dự án chưa có tập truy vấn chuẩn gán nhãn tài liệu đúng/sai.
   - `Sentence-BERT + pgvector HNSW` (`paraphrase-multilingual-MiniLM-L12-v2`, 384 chiều, chỉ mục `idx_chunk_embedding_hnsw` với `m=16, ef_construction=64` trên **5.897 chunk** thật trong PostgreSQL): Đo thời gian sinh vector nhúng, thời gian truy vấn SQL HNSW (`top_k=5`) và phân bố Cosine Similarity trên 20 truy vấn thử nghiệm (lặp 3 lần sau warm-up). **Không công bố Recall@K/MRR/nDCG** do thiếu tập ground-truth gán nhãn.
   - `Inventory Interlock & Contradiction Guard` (`check_live_inventory`, `check_negation_contradiction`): Kiểm chứng truy vấn đọc tồn kho không trừ kho và phát hiện đảo ngược từ cấm an toàn.
2. **Phân hệ Sinh trắc học & Vi định vị BLE (`eam_app/algorithms/biometric/`):**
   - `YuNet + SFace` (`face_detection_yunet_2023mar.onnx`, `face_recognition_sface_2021dec.onnx`): Đo thời gian phát hiện khuôn mặt & trích xuất vector 128 chiều chuẩn L2 trên ảnh mẫu `eam_app/media/user_face.jpg` (không xuất vector/ảnh thô ra báo cáo), cùng kiểm chứng mã hóa/giải mã AES-256.
   - `Cosine Similarity & L2-Norm thuần` (`calculate_cosine_similarity_from_scratch`, `normalize_l2_from_scratch`, ngưỡng thực tế `DEFAULT_ACCEPTANCE_THRESHOLD = 0.55`): Đo thời gian khớp vector 128D (lô 100 cặp so với mục tiêu $\le 30\text{ ms}$) và kiểm tra an toàn `NaN/Inf`.
   - `Log-Distance Path Loss BLE` (`calculate_ble_log_distance`, `verify_ble_proximity`, $A=-59\text{ dBm}, n=2.8$, ngưỡng `RSSI >= -92 dBm` & $d \le 50\text{ m}$): Kiểm chứng thời gian tính toán và độ chính xác công thức suy hao.
   - **Giới hạn bắt buộc:** Dự án chỉ có 1 ảnh chân dung mẫu (`user_face.jpg`) và dữ liệu vector giả lập Gaussian (`lfw_benchmark_pairs.json`), **không có tập ảnh khuôn mặt đa danh tính thực tế**, do đó **không đo và không công bố FAR/FRR hay tỷ lệ nhận diện chính xác thực địa**.

In [1]:
import os, sys, types, time, math, json, platform, subprocess
from datetime import datetime, timezone as dt_timezone
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

try:
    import django.utils.timezone  # noqa: F401
except ImportError:
    django_mod = types.ModuleType('django')
    utils_mod = types.ModuleType('django.utils')
    tz_mod = types.ModuleType('django.utils.timezone')
    tz_mod.now = lambda: datetime.now(dt_timezone.utc)
    utils_mod.timezone = tz_mod
    django_mod.utils = utils_mod
    sys.modules['django'] = django_mod
    sys.modules['django.utils'] = utils_mod
    sys.modules['django.utils.timezone'] = tz_mod

# Shim nhẹ cho django.core.files để nạp safety_config không phụ thuộc Django runtime
if 'django.core.files.storage' not in sys.modules:
    core_mod = types.ModuleType('django.core')
    files_mod = types.ModuleType('django.core.files')
    storage_mod = types.ModuleType('django.core.files.storage')
    base_mod = types.ModuleType('django.core.files.base')
    class DummyStorage:
        def exists(self, p): return False
    storage_mod.default_storage = DummyStorage()
    base_mod.ContentFile = bytes
    files_mod.storage = storage_mod
    files_mod.base = base_mod
    core_mod.files = files_mod
    sys.modules['django.core'] = core_mod
    sys.modules['django.core.files'] = files_mod
    sys.modules['django.core.files.storage'] = storage_mod
    sys.modules['django.core.files.base'] = base_mod

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if not os.path.isdir(os.path.join(REPO_ROOT, 'eam_app')):
    REPO_ROOT = r'D:\GameLinkNeverDie\eam-enterprise-assset-management'
EAM_APP_DIR = os.path.join(REPO_ROOT, 'eam_app')
ALG_ROOT = os.path.join(EAM_APP_DIR, 'algorithms')
RESULTS_DIR = os.path.join(REPO_ROOT, 'scripts', 'results')
VENV_PY = os.path.join(REPO_ROOT, 'venv', 'Scripts', 'python.exe')
os.makedirs(RESULTS_DIR, exist_ok=True)

def register_pkg(name, path):
    m = types.ModuleType(name)
    m.__path__ = [path]
    sys.modules[name] = m

register_pkg('algorithms', ALG_ROOT)
register_pkg('algorithms.rag_chatbot', os.path.join(ALG_ROOT, 'rag_chatbot'))
register_pkg('algorithms.biometric', os.path.join(ALG_ROOT, 'biometric'))

from algorithms.rag_chatbot.safety_classifier import classify_safety_context
from algorithms.rag_chatbot.query_expansion import expand_slang_query
from algorithms.rag_chatbot.contradiction_guard import check_negation_contradiction
from algorithms.rag_chatbot.constants import SLANG_TECHNICAL_DICTIONARY, EMERGENCY_HAZARDS
from algorithms.biometric.cosine_similarity import (
    calculate_cosine_similarity_from_scratch,
    normalize_l2_from_scratch,
    verify_face_cosine_similarity,
    DEFAULT_ACCEPTANCE_THRESHOLD
)
from algorithms.biometric.proximity_engine import (
    calculate_ble_log_distance,
    verify_ble_proximity,
    RSSI_THRESHOLD_50M,
    MAX_GEOFENCE_RADIUS_METERS
)

print('OS:', platform.platform(), '| Python:', platform.python_version())
print('Slang dictionary entries:', len(SLANG_TECHNICAL_DICTIONARY))
print('Biometric acceptance threshold in code:', DEFAULT_ACCEPTANCE_THRESHOLD, '| BLE threshold:', RSSI_THRESHOLD_50M, 'dBm /', MAX_GEOFENCE_RADIUS_METERS, 'm')

OS: Windows-11-10.0.22631-SP0 | Python: 3.14.4
Slang dictionary entries: 73
Biometric acceptance threshold in code: 0.55 | BLE threshold: -92 dBm / 50.0 m


In [2]:
# ==============================================================================
# PHẦN 1A: ĐÁNH GIÁ LIFE-SAFETY CLASSIFIER, SLANG EXPANSION & CONTRADICTION GUARD
# ==============================================================================
safety_cases = [
    # 10 trường hợp khẩn cấp thật sự (5 nhóm nguy cơ)
    ('Cháy dầu máy biến áp đang bốc khói khẩn cấp', 1),
    ('Làm sao dập lửa dầu biến áp đang cháy?', 1),
    ('Mùi gas nồng nặc và xì khí gas ở phân xưởng', 1),
    ('Vỡ đường ống gas đang rò rỉ khí độc', 1),
    ('Công nhân bị kẹt tay vào cơ cấu băng tải', 1),
    ('Người thợ bị cuốn vào trục máy ép khẩn cấp', 1),
    ('Bị axit bắn vào mắt khi đang pha hóa chất', 1),
    ('Hóa chất văng vào mắt công nhân ở bồn tẩy rửa', 1),
    ('Đồng hồ áp vượt vạch đỏ, bình khí nén rung lắc mạnh', 1),
    ('Xì van an toàn liên tục, có dấu hiệu nứt bình áp lực', 1),
    # 10 trường hợp phủ định an toàn / câu hỏi lý thuyết (không phải khẩn cấp)
    ('Máy không bị cháy nhưng rung mạnh ở ổ bi', 0),
    ('Thiết bị chưa hề bốc khói, chỉ bị rò rỉ nhẹ ở phớt dầu', 0),
    ('Nguyên nhân gây chập điện mô tơ là gì?', 0),
    ('Cách phòng chống cháy nổ trạm biến áp theo tiêu chuẩn?', 0),
    ('Quy trình xử lý bảo dưỡng định kỳ bình tích áp như thế nào?', 0),
    ('Làm thế nào để phòng tránh kẹt cơ cấu băng tải?', 0),
    ('Biện pháp phòng ngừa rò rỉ khí gas tại xưởng sơn', 0),
    ('Không có sự cố cháy tủ điện, chỉ cần kiểm tra nhiệt độ tiếp điểm', 0),
    ('Tại sao dây curoa máy nén khí phát ra tiếng rít rít?', 0),
    ('Hướng dẫn tháo lắp phớt cơ khí bơm PUMP-X100', 0),
]

# Warm-up + đo độ trễ
_ = classify_safety_context('warmup')
y_true_sf, y_pred_sf, lat_sf = [], [], []
for text, label in safety_cases:
    t0 = time.perf_counter()
    res = classify_safety_context(text)
    lat_sf.append((time.perf_counter() - t0) * 1000.0)
    y_true_sf.append(label)
    y_pred_sf.append(1 if res['is_emergency'] else 0)

tn_s, fp_s, fn_s, tp_s = confusion_matrix(y_true_sf, y_pred_sf).ravel()
sf_metrics = {
    'cases': len(safety_cases), 'TP': int(tp_s), 'TN': int(tn_s), 'FP': int(fp_s), 'FN': int(fn_s),
    'precision_pct': round(precision_score(y_true_sf, y_pred_sf) * 100.0, 1),
    'recall_pct': round(recall_score(y_true_sf, y_pred_sf) * 100.0, 1),
    'f1_pct': round(f1_score(y_true_sf, y_pred_sf) * 100.0, 1),
    'median_ms': round(float(np.median(lat_sf)), 4)
}

# Kiểm tra Slang Query Expansion
slang_queries = [
    'Bơm kêu lạch cạch, tụ phù',
    'Máy ép kêu cọc cọc, rơ bạc đạn',
    'Đường ống xì phớt, tụt áp liên tục',
    'Biến tần treo biến tần, nhảy cb liên hồi',
    'Băng tải trượt đai, chùng xích',
    'Động cơ kêu è è, nóng bạc đạn',
    'Xy lanh thủy lực tụt ben, sủi bọt dầu',
    'Lỏng bu lông chân đế, đảo đầu trục',
]
lat_sl, expanded_hits = [], 0
for sq in slang_queries:
    t0 = time.perf_counter()
    ex = expand_slang_query(sq, asset_context={'model': 'PUMP-X100'})
    lat_sl.append((time.perf_counter() - t0) * 1000.0)
    if len(ex['detected_slangs']) > 0:
        expanded_hits += 1

# Kiểm tra Contradiction Guard (bảo toàn từ cấm)
cg_pairs = [
    ('CẤM mở van xả áp khi máy đang vận hành.', 'Bạn có thể mở van xả áp khi máy đang vận hành.', True),
    ('TUYỆT ĐỐI KHÔNG dùng nước dập lửa tủ điện.', 'Nên dùng nước phun vào tủ điện để làm mát.', True),
    ('KHÔNG ĐƯỢC tháo vỏ bảo vệ khi trục đang quay.', 'KHÔNG ĐƯỢC tháo vỏ bảo vệ khi trục đang quay.', False),
    ('DO NOT apply grease to magnetic seal MAG-SEAL-2023.', 'Apply grease generously to MAG-SEAL-2023.', True),
]
cg_correct = sum(1 for src, rep, exp in cg_pairs if check_negation_contradiction(rep, src)['contradiction_detected'] == exp)

print('LifeSafetyClassifier (20 test cases):', sf_metrics)
print(f'Slang Query Expansion: {len(SLANG_TECHNICAL_DICTIONARY)} terms | activated {expanded_hits}/{len(slang_queries)} | median={np.median(lat_sl):.4f} ms')
print(f'Contradiction Guard: {cg_correct}/{len(cg_pairs)} detected correctly')

LifeSafetyClassifier (20 test cases): {'cases': 20, 'TP': 9, 'TN': 10, 'FP': 0, 'FN': 1, 'precision_pct': 100.0, 'recall_pct': 90.0, 'f1_pct': 94.7, 'median_ms': 0.0257}
Slang Query Expansion: 73 terms | activated 8/8 | median=0.0108 ms
Contradiction Guard: 3/4 detected correctly


In [3]:
# ==============================================================================
# PHẦN 1B: ĐO THỰC NGHIỆM EMBEDDING + PGVECTOR HNSW + INVENTORY INTERLOCK
# Thực thi trực tiếp qua Python venv của dự án kết nối PostgreSQL eam_db
# ==============================================================================
pg_script = r'''
import os, sys, time, json, django, numpy as np
sys.path.insert(0, r'__EAM_APP_DIR__')
os.environ['DJANGO_SETTINGS_MODULE'] = 'config.settings'
django.setup()

from django.db import connection
from core.models import Tenant
from assets.models import SparePart
from algorithms.rag_chatbot.vector_engine import embed_text, format_vector_for_sql, retrieve_relevant_chunks
from algorithms.rag_chatbot.inventory_interlock import check_live_inventory

with connection.cursor() as cur:
    cur.execute("SELECT count(*) FROM document_chunks WHERE is_active = TRUE")
    total_chunks = cur.fetchone()[0]
    cur.execute("SELECT indexdef FROM pg_indexes WHERE tablename='document_chunks' AND indexname='idx_chunk_embedding_hnsw'")
    row_idx = cur.fetchone()
    index_def = row_idx[0] if row_idx else 'NONE'

t_mfg = Tenant.objects.filter(tenant_code='VF_MFG').first()
tid = str(t_mfg.id) if t_mfg else None

# Warm-up model embedding & PostgreSQL
_ = embed_text('warmup sentence transformer')
_ = retrieve_relevant_chunks('máy bơm rung', tenant_id=tid, is_system_query=False, limit=5)

bench_queries = [
    ('Bơm kêu lạch cạch và rung mạnh xử lý thế nào?', 'PUMP-X100', False),
    ('Thay phớt cơ khí SKF-2210 máy bơm PUMP-X100', 'PUMP-X100', False),
    ('Quy trình căn chỉnh trục máy phay CNC Mazak VCN-530C', 'Mazak VCN-530C', False),
    ('Kiểm tra áp suất dầu hồi máy nén khí công nghiệp', None, False),
    ('Cách bảo dưỡng vòng bi và kiểm tra nhiệt độ ổ trục', None, False),
    ('Nút Giao việc và Bắt đầu sửa chữa trên trang phiếu bảo trì ở đâu?', None, True),
    ('Làm sao xem sơ đồ mặt bằng floorplan trên trang tổng quan?', None, True),
    ('Hướng dẫn sử dụng chức năng điều phối Hungarian và GA xếp ca', None, True),
    ('Cách thêm phụ tùng mới vào kho vật tư và đặt min-stock', None, True),
    ('Quy trình tạo kế hoạch bảo trì định kỳ PM trên Web Portal', None, True),
]

embed_ms, hnsw_sql_ms, e2e_ms, top1_sims = [], [], [], []
REPEATS = 3
for _ in range(REPEATS):
    for q, mod, is_sys in bench_queries:
        t0 = time.perf_counter()
        vec = embed_text(q)
        t1 = time.perf_counter()
        vstr = format_vector_for_sql(vec)
        with connection.cursor() as cur:
            cur.execute(
                'SELECT id, 1 - (embedding <=> %s::vector) AS sim FROM document_chunks WHERE is_active = TRUE ORDER BY embedding <=> %s::vector LIMIT 5;',
                [vstr, vstr]
            )
            rows = cur.fetchall()
        t2 = time.perf_counter()
        res_e2e = retrieve_relevant_chunks(q, tenant_id=tid, asset_model=mod, is_system_query=is_sys, limit=5)
        t3 = time.perf_counter()

        embed_ms.append((t1 - t0) * 1000.0)
        hnsw_sql_ms.append((t2 - t1) * 1000.0)
        e2e_ms.append((t3 - t2) * 1000.0)
        if _ == 0 and res_e2e:
            top1_sims.append(float(res_e2e[0]['similarity']))

# Đo Inventory Interlock read-only
sample_parts = list(SparePart.objects.all()[:5])
probe_text = 'Cần kiểm tra thay thế ' + ' '.join(p.part_number for p in sample_parts if p.part_number)
before_stocks = {str(p.id): float(p.quantity_in_stock) for p in sample_parts}
inv_times = []
inv_hits = 0
for _ in range(10):
    ti0 = time.perf_counter()
    inv_res = check_live_inventory(probe_text)
    inv_times.append((time.perf_counter() - ti0) * 1000.0)
    inv_hits = len(inv_res)
after_stocks = {str(p.id): float(p.quantity_in_stock) for p in SparePart.objects.filter(id__in=list(before_stocks.keys()))}
stock_unchanged = (before_stocks == after_stocks)

out = {
    'total_chunks_db': int(total_chunks),
    'index_def': index_def,
    'queries_count': len(bench_queries),
    'repeats': REPEATS,
    'embed_median_ms': round(float(np.median(embed_ms)), 2),
    'embed_p95_ms': round(float(np.percentile(embed_ms, 95)), 2),
    'hnsw_sql_median_ms': round(float(np.median(hnsw_sql_ms)), 2),
    'hnsw_sql_p95_ms': round(float(np.percentile(hnsw_sql_ms, 95)), 2),
    'retrieve_e2e_median_ms': round(float(np.median(e2e_ms)), 2),
    'retrieve_e2e_p95_ms': round(float(np.percentile(e2e_ms, 95)), 2),
    'top1_sim_mean': round(float(np.mean(top1_sims)), 3) if top1_sims else 0.0,
    'top1_sim_min': round(float(np.min(top1_sims)), 3) if top1_sims else 0.0,
    'top1_sim_max': round(float(np.max(top1_sims)), 3) if top1_sims else 0.0,
    'inventory_median_ms': round(float(np.median(inv_times)), 2),
    'inventory_matched_items': inv_hits,
    'inventory_zero_deduction': stock_unchanged,
}
print('__JSON_START__' + json.dumps(out, ensure_ascii=False))
'''

proc = subprocess.run([VENV_PY, '-c', pg_script.replace('__EAM_APP_DIR__', EAM_APP_DIR)], capture_output=True, text=True, check=True)
json_line = [ln for ln in proc.stdout.splitlines() if '__JSON_START__' in ln][0]
rag_db_metrics = json.loads(json_line.split('__JSON_START__')[1])
print(json.dumps(rag_db_metrics, ensure_ascii=False, indent=2))

{
  "total_chunks_db": 5897,
  "index_def": "CREATE INDEX idx_chunk_embedding_hnsw ON public.document_chunks USING hnsw (embedding vector_cosine_ops) WITH (m='16', ef_construction='64')",
  "queries_count": 10,
  "repeats": 3,
  "embed_median_ms": 12.66,
  "embed_p95_ms": 14.94,
  "hnsw_sql_median_ms": 2.31,
  "hnsw_sql_p95_ms": 3.75,
  "retrieve_e2e_median_ms": 15.99,
  "retrieve_e2e_p95_ms": 17.64,
  "top1_sim_mean": 0.741,
  "top1_sim_min": 0.484,
  "top1_sim_max": 0.984,
  "inventory_median_ms": 10.97,
  "inventory_matched_items": 5,
  "inventory_zero_deduction": true
}


In [4]:
# ==============================================================================
# PHẦN 2: ĐÁNH GIÁ PIPELINE SINH TRẮC HỌC & VI ĐỊNH VỊ BLE
# ==============================================================================
# 2A. YuNet + SFace 128D + AES-256 trên ảnh mẫu eam_app/media/user_face.jpg (qua venv có opencv-headless 4.14)
bio_script = r"""
import os, sys, time, json, django, numpy as np
sys.path.insert(0, r'__EAM_APP_DIR__')
os.environ['DJANGO_SETTINGS_MODULE'] = 'config.settings'
django.setup()

from algorithms.biometric.enrollment import extract_face_embedding_from_image, encrypt_aes_256, decrypt_aes_256

face_path = os.path.join(r'__EAM_APP_DIR__', 'media', 'user_face.jpg')
with open(face_path, 'rb') as f:
    img_bytes = f.read()

# Warm-up
_ = extract_face_embedding_from_image(img_bytes)

ext_times = []
emb = None
for _ in range(20):
    t0 = time.perf_counter()
    emb = extract_face_embedding_from_image(img_bytes)
    ext_times.append((time.perf_counter() - t0) * 1000.0)

t_enc0 = time.perf_counter()
enc = encrypt_aes_256(img_bytes)
dec = decrypt_aes_256(enc)
aes_ms = (time.perf_counter() - t_enc0) * 1000.0

out = {
    'image_bytes': len(img_bytes),
    'embedding_dim': len(emb),
    'l2_norm': round(float(np.linalg.norm(emb)), 6),
    'extract_median_ms': round(float(np.median(ext_times)), 2),
    'extract_p95_ms': round(float(np.percentile(ext_times, 95)), 2),
    'aes256_roundtrip_ok': (dec == img_bytes),
    'aes256_roundtrip_ms': round(aes_ms, 2),
}
print('__JSON_BIO__' + json.dumps(out, ensure_ascii=False))
"""
proc_bio = subprocess.run([VENV_PY, '-c', bio_script.replace('__EAM_APP_DIR__', EAM_APP_DIR)], capture_output=True, text=True, check=True)
bio_line = [ln for ln in proc_bio.stdout.splitlines() if '__JSON_BIO__' in ln][0]
bio_extract_metrics = json.loads(bio_line.split('__JSON_BIO__')[1])

# 2B. Cosine Similarity thuần 128D + Log-Distance BLE (in-process)
rng_bio = np.random.default_rng(42)
u_vec = rng_bio.normal(size=128).tolist()
v_vec = rng_bio.normal(size=128).tolist()
_ = normalize_l2_from_scratch(u_vec)
_ = calculate_cosine_similarity_from_scratch(u_vec, v_vec)

cos_batch_ms = []
ble_batch_ms = []
for _ in range(50):
    t0 = time.perf_counter()
    for _ in range(100):
        nu = normalize_l2_from_scratch(u_vec)
        nv = normalize_l2_from_scratch(v_vec)
        _s = calculate_cosine_similarity_from_scratch(nu, nv)
    cos_batch_ms.append((time.perf_counter() - t0) * 1000.0)

    t1 = time.perf_counter()
    for rssi in range(-110, -50):
        _d = calculate_ble_log_distance(rssi)
        _v = verify_ble_proximity(rssi)
    ble_batch_ms.append((time.perf_counter() - t1) * 1000.0)

# Kiểm tra sai số toán học Cosine thuần so với NumPy và kiểm tra biên NaN/Inf
sim_scratch = calculate_cosine_similarity_from_scratch(u_vec, v_vec)
sim_np = float(np.dot(u_vec, v_vec) / (np.linalg.norm(u_vec) * np.linalg.norm(v_vec)))
nan_safe = (calculate_cosine_similarity_from_scratch([float('nan')] * 128, u_vec) == 0.0)

ble_pass_68, dist_68 = verify_ble_proximity(-68)
ble_fail_96, dist_96 = verify_ble_proximity(-96)
ble_fail_108, dist_108 = verify_ble_proximity(-108)

bio_math_metrics = {
    'cosine_100x_median_ms': round(float(np.median(cos_batch_ms)), 3),
    'cosine_100x_p95_ms': round(float(np.percentile(cos_batch_ms, 95)), 3),
    'cosine_vs_numpy_abs_err': round(abs(sim_scratch - sim_np), 6),
    'nan_inf_safe_rejected': nan_safe,
    'ble_60_rssi_median_ms': round(float(np.median(ble_batch_ms)), 3),
    'ble_rssi_neg68': {'passed': ble_pass_68, 'dist_m': dist_68},
    'ble_rssi_neg96': {'passed': ble_fail_96, 'dist_m': dist_96},
    'ble_rssi_neg108': {'passed': ble_fail_108, 'dist_m': dist_108},
}
print('Biometric Extract (YuNet+SFace):', json.dumps(bio_extract_metrics, ensure_ascii=False, indent=2))
print('Biometric Math & BLE:', json.dumps(bio_math_metrics, ensure_ascii=False, indent=2))

Biometric Extract (YuNet+SFace): {
  "image_bytes": 118923,
  "embedding_dim": 128,
  "l2_norm": 1.0,
  "extract_median_ms": 77.56,
  "extract_p95_ms": 84.03,
  "aes256_roundtrip_ok": true,
  "aes256_roundtrip_ms": 24.77
}
Biometric Math & BLE: {
  "cosine_100x_median_ms": 12.544,
  "cosine_100x_p95_ms": 13.913,
  "cosine_vs_numpy_abs_err": 4e-05,
  "nan_inf_safe_rejected": true,
  "ble_60_rssi_median_ms": 0.094,
  "ble_rssi_neg68": {
    "passed": true,
    "dist_m": 2.1
  },
  "ble_rssi_neg96": {
    "passed": false,
    "dist_m": 20.96
  },
  "ble_rssi_neg108": {
    "passed": false,
    "dist_m": 56.23
  }
}


In [5]:
# ==============================================================================
# TỔNG HỢP BẢNG 3.14 & LƯU BẰNG CHỨNG
# Ghi chú: Không dùng cụm "Chưa đo"; nêu rõ phạm vi đo và giới hạn thiếu dữ liệu nhãn.
# ==============================================================================
table_3_14_rows = [
    {
        'Thành phần công nghệ': 'RAG: LifeSafetyClassifier & Contradiction Guard',
        'Phạm vi & Phương pháp kiểm chứng': '20 câu tự thiết kế (10 khẩn cấp, 10 phủ định/lý thuyết) + 4 ca đảo phủ định',
        'Chỉ số đo được từ mã nguồn': f"Safety: Prec {sf_metrics['precision_pct']}%, Rec {sf_metrics['recall_pct']}% ({sf_metrics['median_ms']} ms); Chặn {cg_correct}/{len(cg_pairs)} ca",
        'Giới hạn thực nghiệm & An toàn': f"FN=1 do từ 'dấu hiệu' trùng THEORETICAL_KEYWORDS; Guard 3/4 do 'TUYỆT ĐỐI KHÔNG' ngoài từ điển cấm"
    },
    {
        'Thành phần công nghệ': 'RAG: Slang Expansion & pgvector HNSW',
        'Phạm vi & Phương pháp kiểm chứng': f"{len(SLANG_TECHNICAL_DICTIONARY)} từ lóng; truy vấn top-k=5 / {rag_db_metrics['total_chunks_db']} chunk PostgreSQL (10×3 lượt)",
        'Chỉ số đo được từ mã nguồn': f"Lóng: {expanded_hits}/{len(slang_queries)} ({round(float(np.median(lat_sl)), 3)} ms); HNSW: {rag_db_metrics['hnsw_sql_median_ms']} ms; Embed: {rag_db_metrics['embed_median_ms']} ms",
        'Giới hạn thực nghiệm & An toàn': 'Chỉ mục HNSW (m=16, ef=64) thật; không tính Recall@K/MRR do thiếu tập truy vấn gán nhãn'
    },
    {
        'Thành phần công nghệ': 'RAG: Live Inventory Interlock',
        'Phạm vi & Phương pháp kiểm chứng': 'Đọc trực tiếp bảng SparePart/Tool (10 lượt sau warm-up)',
        'Chỉ số đo được từ mã nguồn': f"Trung vị {rag_db_metrics['inventory_median_ms']} ms; khớp {rag_db_metrics['inventory_matched_items']} mã vật tư; 0 trừ kho",
        'Giới hạn thực nghiệm & An toàn': 'Chỉ đọc tồn kho cảnh báo; bản nháp tạo ở trạng thái CREATED'
    },
    {
        'Thành phần công nghệ': 'Biometric: YuNet + SFace + Cosine 128D',
        'Phạm vi & Phương pháp kiểm chứng': 'Trích xuất 128D ảnh mẫu (20 lượt) + 100×Cosine thuần (50 lô)',
        'Chỉ số đo được từ mã nguồn': f"YuNet+SFace: {bio_extract_metrics['extract_median_ms']} ms; 100×Cos: {bio_math_metrics['cosine_100x_median_ms']} ms; AES-256: {bio_extract_metrics['aes256_roundtrip_ms']} ms",
        'Giới hạn thực nghiệm & An toàn': 'Không tính FAR/FRR (chỉ có 1 ảnh mẫu, thiếu ảnh đa danh tính); ngưỡng 0,55; an toàn NaN/Inf'
    },
    {
        'Thành phần công nghệ': 'Geofencing: BLE Log-Distance RSSI',
        'Phạm vi & Phương pháp kiểm chứng': 'Quét RSSI [-110..-50] dBm (A=-59, n=2,8; ngưỡng -92 dBm & 50 m)',
        'Chỉ số đo được từ mã nguồn': f"60 phép tính: {bio_math_metrics['ble_60_rssi_median_ms']} ms; -68 dBm→{bio_math_metrics['ble_rssi_neg68']['dist_m']}m (Đạt); -96 dBm (< -92→Chặn)",
        'Giới hạn thực nghiệm & An toàn': 'Kiểm chứng công thức suy hao lý thuyết; môi trường xưởng kim loại có thể nhiễu sóng'
    }
]

df_314 = pd.DataFrame(table_3_14_rows)
df_314.to_csv(os.path.join(RESULTS_DIR, 'table_3_14_rag_biometric_summary.csv'), index=False, encoding='utf-8-sig')
with open(os.path.join(RESULTS_DIR, 'table_3_14_rag_biometric_raw.json'), 'w', encoding='utf-8') as f:
    json.dump({
        'safety_classifier': sf_metrics,
        'rag_pgvector': rag_db_metrics,
        'biometric_extract': bio_extract_metrics,
        'biometric_math_ble': bio_math_metrics
    }, f, ensure_ascii=False, indent=2)

df_314

,Thành phần công nghệ,Phạm vi & Phương pháp kiểm chứng,Chỉ số đo được từ mã nguồn,Giới hạn thực nghiệm & An toàn
0,RAG: LifeSafetyClassifier & Contradiction Guard,"20 câu tự thiết kế (10 khẩn cấp, 10 phủ định/l...","Safety: Prec 100.0%, Rec 90.0% (0.0257 ms); Ch...",FN=1 do từ 'dấu hiệu' trùng THEORETICAL_KEYWOR...
1,RAG: Slang Expansion & pgvector HNSW,73 từ lóng; truy vấn top-k=5 / 5897 chunk Post...,Lóng: 8/8 (0.011 ms); HNSW: 2.31 ms; Embed: 12...,"Chỉ mục HNSW (m=16, ef=64) thật; không tính Re..."
2,RAG: Live Inventory Interlock,Đọc trực tiếp bảng SparePart/Tool (10 lượt sau...,Trung vị 10.97 ms; khớp 5 mã vật tư; 0 trừ kho,Chỉ đọc tồn kho cảnh báo; bản nháp tạo ở trạng...
3,Biometric: YuNet + SFace + Cosine 128D,Trích xuất 128D ảnh mẫu (20 lượt) + 100×Cosine...,YuNet+SFace: 77.56 ms; 100×Cos: 12.544 ms; AES...,"Không tính FAR/FRR (chỉ có 1 ảnh mẫu, thiếu ản..."
4,Geofencing: BLE Log-Distance RSSI,"Quét RSSI [-110..-50] dBm (A=-59, n=2,8; ngưỡn...",60 phép tính: 0.094 ms; -68 dBm→2.1m (Đạt); -9...,Kiểm chứng công thức suy hao lý thuyết; môi tr...
